<a href="https://colab.research.google.com/github/shin584/project/blob/1D_models/Cas12a_model4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv1D, Dense, BatchNormalization,
    Dropout, Bidirectional, LSTM, GlobalAveragePooling1D, LeakyReLU
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import os
from google.colab import drive

drive.mount('/content/drive')
base_path = '' # 따로 입력해주세요
file_name = '' # 따로 입력해주세요
file_path = os.path.join(base_path, file_name)

if os.path.exists(file_path):
    print(f"파일을 찾았습니다! 경로: {file_path}")
else:
    print(f"파일을 찾을 수 없습니다: {file_path}")


def one_hot_encode(sequence):
    mapping = {'A': [1,0,0,0], 'C': [0,1,0,0], 'G': [0,0,1,0], 'T': [0,0,0,1]}
    sequence = sequence.upper()
    return np.array([mapping.get(base, [0,0,0,0]) for base in sequence])


def extract_seq_features(seq):
    protospacer = seq[8:31]

    gc_total = (protospacer.count('G') + protospacer.count('C')) / 23
    gc_seed  = (protospacer[-8:].count('G') + protospacer[-8:].count('C')) / 8

    dinuc_map = {a+b: i for i, (a, b) in enumerate(
        [(x, y) for x in 'ACGT' for y in 'ACGT']
    )}
    dinuc_features = np.zeros(22 * 16, dtype=np.float32)
    for i in range(22):
        pair = protospacer[i:i+2]
        if pair in dinuc_map:
            dinuc_features[i * 16 + dinuc_map[pair]] = 1.0

    return np.array([gc_total, gc_seed], dtype=np.float32), dinuc_features


def load_cas12a_dataset(filepath):
    print("Cas12a 데이터 로딩 중...")

    if filepath.endswith(('.csv', '.txt')):
        df = pd.read_csv(filepath)
        if len(df.columns) < 2:
            df = pd.read_csv(filepath, sep='\t')
    elif filepath.endswith('.tsv'):
        df = pd.read_csv(filepath, sep='\t')
    elif filepath.endswith(('.xls', '.xlsx')):
        df = pd.read_excel(filepath)
    else:
        raise ValueError("Unsupported file format.")

    print(f"컬럼 목록: {list(df.columns)}")
    print(f"전체 행 수: {len(df)}")

    seq_col    = '34 bp synthetic target and target context sequence\n(4 bp + PAM + 23 bp protospacer + 3 bp)'
    target_col = 'Indel freqeuncy\n(Background substracted, %)'

    original_len = len(df)
    df[target_col] = pd.to_numeric(df[target_col], errors='coerce')
    df = df.dropna(subset=[seq_col, target_col])
    print(f"결측치 제거: {original_len} -> {len(df)}개")

    X_seq, X_feat, y = [], [], []

    for _, row in df.iterrows():
        seq   = str(row[seq_col]).strip().upper()
        score = float(row[target_col])

        if len(seq) > 34:
            seq = seq[:34]
        elif len(seq) < 34:
            seq = seq + ('N' * (34 - len(seq)))

        gc_feat, dinuc_feat = extract_seq_features(seq)
        extra = np.concatenate([gc_feat, dinuc_feat])

        X_seq.append(one_hot_encode(seq))
        X_feat.append(extra)
        y.append(score / 100.0)

    X_seq  = np.array(X_seq,  dtype=np.float32)
    X_feat = np.array(X_feat, dtype=np.float32)
    y      = np.array(y,      dtype=np.float32)
    y      = np.clip(y, 0, 1)

    print(f"최종 데이터: X_seq={X_seq.shape}, X_feat={X_feat.shape}, y={y.shape}")
    print(f"타겟 범위  : {y.min()*100:.1f}% ~ {y.max()*100:.1f}%")
    return X_seq, X_feat, y



def create_cas12a_model():
    seq_input = Input(shape=(34, 4), name='Input_Seq')

    x1 = Conv1D(64, kernel_size=4,  padding='same', name='Conv_k4')(seq_input)
    x1 = BatchNormalization(name='BN_k4')(x1)
    x1 = LeakyReLU(negative_slope=0.1, name='LReLU_k4')(x1)

    x2 = Conv1D(64, kernel_size=8,  padding='same', name='Conv_k8')(seq_input)
    x2 = BatchNormalization(name='BN_k8')(x2)
    x2 = LeakyReLU(negative_slope=0.1, name='LReLU_k8')(x2)

    x3 = Conv1D(64, kernel_size=15, padding='same', name='Conv_k15')(seq_input)
    x3 = BatchNormalization(name='BN_k15')(x3)
    x3 = LeakyReLU(negative_slope=0.1, name='LReLU_k15')(x3)

    x = tf.keras.layers.Concatenate(name='Concat_conv')([x1, x2, x3])
    x = Dropout(0.4, name='Drop1')(x)

    x = Bidirectional(LSTM(32, return_sequences=True), name='BiLSTM')(x)
    x = Dropout(0.4, name='Drop2')(x)
    x = GlobalAveragePooling1D(name='GAP')(x)

    feat_input = Input(shape=(354,), name='Input_Feat')
    f = Dense(64, activation='relu', name='Feat_Dense')(feat_input)
    f = Dropout(0.3, name='Drop_feat')(f)

    x = tf.keras.layers.Concatenate(name='Concat_all')([x, f])
    x = Dense(64, activation='relu', name='Dense1')(x)
    x = Dropout(0.4, name='Drop3')(x)
    output = Dense(1, activation='sigmoid', name='Output')(x)

    model = Model(inputs=[seq_input, feat_input], outputs=output)
    model.compile(optimizer=Adam(learning_rate=0.0005), loss='mse', metrics=['mae'])
    return model



def train_5fold(X_seq, X_feat, y):
    model_save_dir = os.path.join(base_path, 'model_weights')
    os.makedirs(model_save_dir, exist_ok=True)

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    fold_results = []
    fold_models  = []

    print("\n===== 5-Fold 학습 시작 =====\n")

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_seq)):
        print(f"---------- Fold {fold+1} / 5 ----------")

        X_seq_tr,  X_seq_val  = X_seq[train_idx],  X_seq[val_idx]
        X_feat_tr, X_feat_val = X_feat[train_idx], X_feat[val_idx]
        y_train,   y_val      = y[train_idx],       y[val_idx]

        model = create_cas12a_model()

        early_stopping = EarlyStopping(
            monitor='val_loss',
            patience=15,
            restore_best_weights=True
        )
        reduce_lr = ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=10,
            min_lr=1e-6
        )

        model.fit(
            [X_seq_tr, X_feat_tr], y_train,
            validation_data=([X_seq_val, X_feat_val], y_val),
            epochs=100,
            batch_size=32,
            callbacks=[early_stopping, reduce_lr],
            verbose=1
        )

        save_path = os.path.join(model_save_dir, f'cas12a_fold{fold+1}.keras')
        model.save(save_path)
        print(f"저장 완료: {save_path}")

        y_pred     = model.predict([X_seq_val, X_feat_val], verbose=0).flatten()
        y_true_pct = y_val  * 100
        y_pred_pct = y_pred * 100

        mae     = mean_absolute_error(y_true_pct, y_pred_pct)
        r2      = r2_score(y_true_pct, y_pred_pct)
        pearson = pearsonr(y_true_pct, y_pred_pct)[0]

        fold_results.append({'fold': fold+1, 'mae': mae, 'r2': r2, 'pearson': pearson})
        fold_models.append(model)


    return fold_models, fold_results


def evaluate_ensemble(fold_models, X_seq, X_feat, y):
    print("\n===== 앙상블 최종 성능 =====")

    preds = np.array([
        m.predict([X_seq, X_feat], verbose=0).flatten()
        for m in fold_models
    ])
    y_pred_ensemble = np.mean(preds, axis=0)

    y_true_pct = y               * 100
    y_pred_pct = y_pred_ensemble * 100

    mae      = mean_absolute_error(y_true_pct, y_pred_pct)
    r2       = r2_score(y_true_pct, y_pred_pct)
    pearson  = pearsonr(y_true_pct, y_pred_pct)[0]
    residuals = y_true_pct - y_pred_pct

    print(f"  MAE      : {mae:.2f}%")
    print(f"\n구간별 MAE")
    print(f"  저효율(0~20%)   : {np.mean(np.abs(residuals[y_true_pct < 20])):.2f}%")
    print(f"  중효율(20~60%)  : {np.mean(np.abs(residuals[(y_true_pct >= 20) & (y_true_pct < 60)])):.2f}%")
    print(f"  고효율(60~100%) : {np.mean(np.abs(residuals[y_true_pct >= 60])):.2f}%")

    plt.figure(figsize=(6, 6))
    plt.scatter(y_true_pct, y_pred_pct, alpha=0.2, s=5, color='steelblue')
    plt.plot([0, 100], [0, 100], 'r--', linewidth=1.5, label='Perfect fit')
    plt.title(f"Cas12a Ensemble: Predicted vs Actual\nMAE={mae:.2f}%  R²={r2:.3f}  Pearson={pearson:.3f}")
    plt.xlabel("Actual Indel Frequency (%)")
    plt.ylabel("Predicted Indel Frequency (%)")
    plt.xlim(0, 100); plt.ylim(0, 100)
    plt.legend(); plt.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()



def predict_cas12a(sequence, fold_models):
    """
    입력: 34bp DNA 서열
    출력: 5개 모델 앙상블 raw score (0~1)
    """
    seq = sequence.strip().upper()

    if len(seq) != 34:
        print(f"[Error] 서열 길이가 맞지 않습니다. (현재: {len(seq)}bp / 필요: 34bp)")
        return None

    pam_seq = seq[4:8]
    enc     = one_hot_encode(seq).reshape(1, 34, 4)

    gc_feat, dinuc_feat = extract_seq_features(seq)
    feat = np.concatenate([gc_feat, dinuc_feat]).reshape(1, 354)

    scores = [float(m.predict([enc, feat], verbose=0)[0][0]) for m in fold_models]
    score  = np.mean(scores)

    print(f"서열    : {seq[:4]}[{pam_seq}]{seq[8:]}")
    print(f"PAM     : {pam_seq}")
    print(f"예측 Indel frequency: {score*100:.2f}%")
    print(f"Fold별 점수: {[f'{s:.4f}' for s in scores]}")

    if score > 0.6:
        print("결과: 높은 절단 효율 (High efficiency)")
    elif score > 0.3:
        print("결과: 보통 절단 효율 (Moderate efficiency)")
    else:
        print("결과: 낮은 절단 효율 (Low efficiency)")

    return score  # 0~1 raw score 반환 (파이프라인용)


if __name__ == '__main__':

    if os.path.exists(file_path):
        X_seq, X_feat, y = load_cas12a_dataset(file_path)
        fold_models, fold_results = train_5fold(X_seq, X_feat, y)
        evaluate_ensemble(fold_models, X_seq, X_feat, y)
    else:
        print(f"파일을 찾을 수 없습니다: {file_path}")

    print("\n[추론 테스트]")
    print("-" * 50)
    seq_test = "ATCGTTTAATCGATCGATCGATCGATCGATCGTA"
    predict_cas12a(seq_test, fold_models)

    print("-" * 50)
    seq_error = "ATCG"
    predict_cas12a(seq_error, fold_models)
